In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import matplotlib.pyplot as plt
import numpy as np
import xrUWLCM as xrU
import math
import pandas as pd
import string

#from xhistogram.xarray import histogram
import xarray as xr

import matplotlib as mpl
from os import listdir
from os.path import isfile, join
from scipy.stats import binned_statistic

# pgsdir = r'/net/pr2/projects/plgrid/plgguwicmw/wyniki/dycoms_rf01/' # long-term data storage, but slow (group folder)
pgsdir = r'/net/tscratch/people/plgpdziekan/wyniki/dycoms_rf01/' # SCRATCH contains copies of (some) data from long-term storage; much faster loading
figoutdir = r'/net/people/plgrid/plgpdziekan/wyniki/dycoms_rf01/figs/'
dataoutdir = r'/net/people/plgrid/plgpdziekan/NextGEMS_Sc_lowres/data/'
scratchdir = r'/net/tscratch/people/plgpdziekan/'
#ncoutdir = '/net/people/plgrid/plgpdziekan/wyniki/dycoms_rf01/netcdf/'

# pgsdir = '/home/piotr/praca/NextGEMS/LES_Smg/data/'
# figoutdir = '/home/piotr/praca/NextGEMS/LES_Smg/figs/'
# dataoutdir = '/home/piotr/praca/NextGEMS/LES_Smg/data/'
#scratchdir = '/net/tscratch/people/plgpdziekan/'
#ncoutdir = '/net/people/plgrid/plgpdziekan/wyniki/dycoms_rf01/netcdf/'

# UWLCM data diretories #

In [ ]:
datadir = {}

datadir['dz50m'] = {}
datadir['dz50m']['dx5000m'] = {}
# datadir['dz50m']['dx5000m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta50_dx5000m_dz50m_X250km_longlong_out_blk_1m/"
# datadir['dz50m']['dx5000m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dx5000m_dz50m_X250km_longlong_out_blk_1m/"
# datadir['dz50m']['dx5000m']['Isotropic_Window'] = pgsdir + "dycomsRF01_SMG_SgsDelta50m_dt1_dx5000m_dz50m_X955km_Window_out_blk_1m/"
# datadir['dz50m']['dx5000m']['Anisotropic_Window'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dt1_dx5000m_dz50m_X955km_Window_out_blk_1m/"
datadir['dz50m']['dx5000m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta50_dt1_dx5000m_dz50m_X955km_NoWindow_out_blk_1m/"
datadir['dz50m']['dx5000m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dt1_dx5000m_dz50m_X955km_NoWindow_out_blk_1m/"

datadir['dz20m'] = {}
datadir['dz20m']['dx2000m'] = {}
##datadir['dz20m']['dx2000m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta20_dx2000m_dz20m_X100km_longlong_out_blk_1m/"
datadir['dz20m']['dx2000m']['Isotropic'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta20m_dt1_dx2000m_dz20m_X574km_longlong_out_blk_1m/"
# datadir['dz20m']['dx2000m']['Isotropic_H100'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta20m_dt1_dx2000m_dz20m_X100km_longlong_out_blk_1m/"
# datadir['dz20m']['dx2000m']['Isotropic_tracers'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta20m_dt1_dx2000m_dz20m_X574km_NoWindow_tracers_longlong_out_blk_1m/"
# datadir['dz20m']['dx2000m']['Isotropic_Window_tracers'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta20m_dt1_dx2000m_dz20m_X574km_Window_tracers_longlong_out_blk_1m/"
# datadir['dz20m']['dx2000m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dx2000m_dz20m_X100km_longlong_out_blk_1m/"
# datadir['dz20m']['dx2000m']['Anisotropic'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt1_dx2000m_dz20m_X574km_longlong_out_blk_1m/"
datadir['dz20m']['dx2000m']['Anisotropic'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt1_dx2000m_dz20m_X574km_NoWindow_tracers_longlong_out_blk_1m/" # Actually Anisotropic_tracers
# datadir['dz20m']['dx2000m']['Anisotropic_tracers'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt1_dx2000m_dz20m_X574km_NoWindow_tracers_longlong_out_blk_1m/"
# datadir['dz20m']['dx2000m']['Anisotropic_Window_tracers'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt1_dx2000m_dz20m_X574km_Window_tracers_longlong_out_blk_1m/"

# datadir['dz20m']['dx1000m'] = {}
# datadir['dz20m']['dx1000m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta20_dx1000m_dz20m_X50km_longlong_out_blk_1m/"
# datadir['dz20m']['dx1000m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dx1000m_dz20m_X50km_longlong_out_blk_1m/"

datadir['dz10m'] = {}
# datadir['dz10m']['dx1000m'] = {}
# datadir['dz10m']['dx1000m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta10_dt1_dx1000m_dz10m_X100km_longlong_out_blk_1m/"
# datadir['dz10m']['dx1000m']['Anisotropic'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt0.25_dx1000m_dz10m_X287km_longlong_out_blk_1m/"

# datadir['dz50m']['dx500m'] = {}
# datadir['dz50m']['dx500m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta50_dx500m_dz50m_X250km_longlong_out_blk_1m/"

# datadir['dz30m'] = {}
# datadir['dz30m']['dx300m'] = {}
# datadir['dz30m']['dx300m']['Isotropic_Window_tracers'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta30m_dt0.5_dx300m_dz30m_X86.1km_Window_tracers_longlong_out_blk_1m/"
# datadir['dz30m']['dx300m']['Isotropic_Window_tracers_large'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta30m_dt0.5_dx300m_dz30m_X172.5km_Window_tracers_longlong_out_blk_1m/"
# datadir['dz30m']['dx300m']['Anisotropic_Window_tracers'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt0.5_dx300m_dz30m_X86.1km_Window_tracers_longlong_out_blk_1m/"
# datadir['dz30m']['dx300m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta30m_dt0.5_dx300m_dz30m_X86.1km_out_blk_1m/"
# datadir['dz30m']['dx300m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dt0.5_dx300m_dz30m_X86.1km_out_blk_1m/"

# datadir['dz20m']['dx500m'] = {}
# datadir['dz20m']['dx500m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dx500m_dz20m_X50km_longlong_out_blk_1m/"

datadir['dz10m']['dx100m'] = {}
datadir['dz10m']['dx100m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta10_dx100m_dz10m_X50km_longlong_out_blk_1m/"
datadir['dz10m']['dx100m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dx100m_dz10m_X50km_longlong_out_blk_1m/"
#datadir['dz10m']['dx100m']['AnisoSmgAlong_dt0.5'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dt5e-1_dx100m_dz10m_X50km_longlong_out_blk_1m/"

datadir['dz5m'] = {}
datadir['dz5m']['dx50m'] = {}
datadir['dz5m']['dx50m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta5_dt5e-1_dx50m_dz5m_X25km_longlong_out_blk_1m/"
datadir['dz5m']['dx50m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dt5e-1_dx50m_dz5m_X25km_longlong_out_blk_1m/"
# datadir['dz5m']['dx50m']['Isotropic_Window'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta5_dt0.5_dx50m_dz5m_X25km_Window_longlong_out_blk_1m/"
# datadir['dz5m']['dx50m']['Isotropic_Window_tracers'] = pgsdir + "singu_gh200_dycomsRF01_SMG_SgsDelta5m_dt0.5_dx50m_dz5m_X25km_Window_tracers_longlong_out_blk_1m/"
# datadir['dz5m']['dx50m']['Anisotropic_Window'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt0.5_dx50m_dz5m_X25km_Window_longlong_out_blk_1m/"
# datadir['dz5m']['dx50m']['Anisotropic_Window_tracers'] = pgsdir + "singu_gh200_dycomsRF01_AnisoSmgAlong_dt0.5_dx50m_dz5m_X25km_Window_tracers_longlong_out_blk_1m/"

# datadir['dz2.5m'] = {}
# datadir['dz2.5m']['dx25m'] = {}
# datadir['dz2.5m']['dx25m']['Isotropic'] = pgsdir + "dycomsRF01_SMG_SgsDelta2.5_dt0.25_dx25m_dz2.5m_X12.5km_longlong_out_blk_1m/"
# datadir['dz2.5m']['dx25m']['Anisotropic'] = pgsdir + "dycomsRF01_AnisoSmgAlong_dt0.25_dx25m_dz2.5m_X12.5km_longlong_out_blk_1m/"

In [ ]:
resolution_nicename = {\
    'dx5000m' : r'$\Delta x,y = 5000\, \mathrm{m}$', \
    'dx2000m' : r'$\Delta x,y = 2000\, \mathrm{m}$', \
    'dx1000m' : r'$\Delta x = 1000 \mathrm{m}$', \
    'dx300m' :  r'$\Delta x = 300 \mathrm{m}$', \
    'dx100m' :  r'$\Delta x,y = 100\, \mathrm{m}$', \
    'dx50m' :   r'$\Delta x,y = 50\, \mathrm{m}$', \
    'dx25m' :   r'$\Delta x = 25 \mathrm{m}$', \
    'dz50m' :   r'$\Delta z = 50\, \mathrm{m}$', \
    'dz30m' :   r'$\Delta z = 30 \mathrm{m}$', \
    'dz20m' :   r'$\Delta z = 20\, \mathrm{m}$', \
    'dz10m' :   r'$\Delta z = 10\, \mathrm{m}$', \
    'dz5m' :    r'$\Delta z = 5\, \mathrm{m}$', \
    'dz2.5m' :  r'$\Delta z = 2.5 \mathrm{m}$', \
}

sgs_nicename = {\
    # 'Isotropic' : 'Isotropic Smagorinsky', \
    'Isotropic' : 'Iso', \
    # 'Anisotropic' : 'Anisotropic Smagorinsky', \
    'Anisotropic' : 'Aniso', \
    'Isotropic_tracers' : 'Isotropic Smagorinsky', \
    'Anisotropic_tracers' : 'Anisotropic Smagorinsky', \
    'Isotropic_Window' : 'Isotropic Smagorinsky', \
    'Anisotropic_Window' : 'Anisotropic Smagorinsky', \
    'Isotropic_Window_tracers' : 'Isotropic Smagorinsky', \
    'Anisotropic_Window_tracers' : 'Anisotropic Smagorinsky', \
    'Isotropic_Window_tracers_large' : 'Isotropic Smagorinsky', \
    'Isotropic_large' : 'Isotropic Smagorinsky', \
    'Anisotropic_large' : 'Anisotropic Smagorinsky', \
}

# define simulations to be plotted #

In [ ]:
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']

# dx, dy, SGS, time period in hours for averaging histograms and profiles, plot parameters, coarsening factor, domain size
# wider lines are for 100x anisotropy, narrower for 10x
data_iso = {\
    ('dx5000m', 'dz50m',  'Isotropic') :   ((23,24), {'ls' : '-.', 'color' : colors[0],    'lw' : 1}, 1, 955), \
    # ('dx5000m', 'dz50m',  'Isotropic_Window') :   ((23,24), {'ls' : '-.', 'color' : colors[0],    'lw' : 2.5}, 1, 955), \
    ('dx2000m', 'dz20m',  'Isotropic') :   ((23,24), {'ls' : '-.', 'color' : colors[1],    'lw' : 1}, 1, 574), \
    # ('dx2000m', 'dz20m',  'Isotropic_tracers') :   ((23,24), {'ls' : '-.', 'color' : colors[1],    'lw' : 2}, 1, 574), \
    # ('dx2000m', 'dz20m',  'Isotropic_Window_tracers') :   ((23,24), {'ls' : '-.', 'color' : colors[1],    'lw' : 2.5}, 1, 574), \
    # ('dx300m',  'dz30m',  'Isotropic_Window_tracers') :   ((11,12),   {'ls' : '-.', 'color' : colors[6],    'lw' : 1.5}, 6.66, 86.1), \
    # ('dx300m',  'dz30m',  'Isotropic') :   ((11,12),   {'ls' : '-.', 'color' : colors[6],    'lw' : 1}, 6.66, 86.1), \
    ('dx50m',   'dz5m',   'Isotropic') :   ((6,7),   {'ls' : '-.', 'color' : colors[4],    'lw' : 1}, 40, 25), \
    # ('dx50m',   'dz5m',   'Isotropic_Window_tracers') :   ((4,4.75),   {'ls' : '-.', 'color' : colors[4],    'lw' : 1.5}, 40, 25), \
    
    # #('dx5000m', 'dz50m',  'Isotropic') :   ((23,24), {'ls' : '-.', 'color' : colors[0],    'lw' : 2}, 1, 250), \# ('dx50m',   'dz5m',   'Isotropic_Window') :   ((4,5),   {'ls' : '-.', 'color' : colors[4],    'lw' : 1.5}, 40, 25), \
    # # #('dx2000m', 'dz20m',  'Isotropic') :   ((20,24), {'ls' : '-.', 'color' : colors[1],    'lw' : 2}, 1, 100), \
    # # #('dx2000m', 'dz20m',  'Isotropic') :   ((1.5,2), {'ls' : '-.', 'color' : colors[1],    'lw' : 2}, 1, 100), \
    # # #('dx2000m', 'dz20m',  'Isotropic') :   ((20,24), {'ls' : '-.', 'color' : colors[1],    'lw' : 2}, 1, 574), \
    # #('dx2000m', 'dz20m',  'Isotropic_Window_tracers') :   ((23,23.5), {'ls' : '-.', 'color' : colors[1],    'lw' : 2.5}, 1, 574), \
    # #('dx2000m', 'dz20m',  'Isotropic_H100') :   ((0.5,1), {'ls' : '-.', 'color' : colors[1],    'lw' : 4}, 1, 100), \
    # #('dx1000m', 'dz20m',  'Isotropic') :   ((1.5,2), {'ls' : '-.', 'color' : colors[6],    'lw' : 1.5}, 1, 100), \
    # #('dx1000m', 'dz20m',  'Isotropic') :   ((23,24), {'ls' : '-.', 'color' : colors[6],    'lw' : 1.5}, 2, 50), \
    # #('dx1000m', 'dz10m',  'Isotropic') :   ((20,24), {'ls' : '-.', 'color' : colors[2],    'lw' : 2}, 2, 100), \
    # #('dx1000m', 'dz10m',  'Isotropic') :   ((1.5, 2), {'ls' : '-.', 'color' : colors[2],    'lw' : 2}, 2, 100), \
    # ('dx300m',  'dz30m',  'Isotropic_Window_tracers_large') :   ((11,12),   {'ls' : '-.', 'color' : colors[6],    'lw' : 1}, 6.66, 172.5), \
    ('dx100m',  'dz10m',  'Isotropic') :   ((6,7),   {'ls' : '-.', 'color' : colors[3],    'lw' : 1}, 20, 50), \
    # ('dx100m',  'dz10m',  'Isotropic') :   ((1.5,2),   {'ls' : '-.', 'color' : colors[3],    'lw' : 1}, 20, 50), \
    #('dx50m',   'dz5m',   'Isotropic_Window_tracers') :   ((4,4.5),   {'ls' : '-.', 'color' : colors[4],    'lw' : 1.5}, 40, 25), \
    # ('dx25m',   'dz2.5m', 'Isotropic') :   ((1.2,1.375),   {'ls' : '--', 'color' : colors[5],    'lw' : 1}, 80, 12.5), \
}

data_aniso = {\
    ('dx5000m', 'dz50m',  'Anisotropic') : ((23,24), {'ls' : '--', 'color' : colors[0],    'lw' : 1}, 1, 955), \
    # ('dx5000m', 'dz50m',  'Anisotropic_Window') : ((23,24), {'ls' : '--', 'color' : colors[0],    'lw' : 2.5}, 1, 955), \
    ('dx2000m', 'dz20m',  'Anisotropic') : ((23,24), {'ls' : '--', 'color' : colors[1],    'lw' : 1}, 1, 574), \
    # ('dx2000m', 'dz20m',  'Anisotropic_tracers') : ((23,24), {'ls' : '--', 'color' : colors[1],    'lw' : 2}, 1, 574), \
    # ('dx2000m', 'dz20m',  'Anisotropic_Window_tracers') : ((23,24), {'ls' : '--', 'color' : colors[1],    'lw' : 2.5}, 1, 574), \
    # ('dx300m',  'dz30m',  'Anisotropic_Window_tracers') : ((11,12),   {'ls' : '--', 'color' : colors[6],    'lw' : 1.5}, 6.66, 86.1), \
    # ('dx300m',  'dz30m',  'Anisotropic') : ((11,12),   {'ls' : '--', 'color' : colors[6],    'lw' : 1}, 6.66, 86.1), \
    ('dx50m',   'dz5m',   'Anisotropic') : ((6,7),   {'ls' : '--', 'color' : colors[4],    'lw' : 1}, 40, 25), \
    # ('dx50m',   'dz5m',   'Anisotropic_Window') :   ((3,3.5),   {'ls' : '--', 'color' : colors[4],    'lw' : 1.5}, 40, 25), \
    
    # ('dx5000m', 'dz50m',  'Anisotropic') : ((23,24), {'ls' : '--', 'color' : colors[0],    'lw' : 2}, 1, 250), \
    # # #('dx2000m', 'dz20m',  'Anisotropic') : ((20,24), {'ls' : '--', 'color' : colors[1],    'lw' : 2}, 1, 100), \
    # # #('dx2000m', 'dz20m',  'Anisotropic') : ((1.5,2), {'ls' : '--', 'color' : colors[1],    'lw' : 2}, 1, 100), \
    # # #('dx2000m', 'dz20m',  'Anisotropic_H') : ((0.5,1), {'ls' : '--', 'color' : colors[1],    'lw' : 2}, 1, 100), \
    # # #('dx2000m', 'dz20m',  'Anisotropic_Window_tracers') : ((23,23.5), {'ls' : '--', 'color' : colors[1],    'lw' : 2.5}, 1, 574), \
    # # #('dx1000m', 'dz20m',  'Anisotropic') : ((1.5,2), {'ls' : '--', 'color' : colors[6],    'lw' : 1.5}, 1, 100), \
    # # #('dx1000m', 'dz20m',  'Anisotropic') : ((23,24), {'ls' : '--', 'color' : colors[6],    'lw' : 1.5}, 2, 50), \
    # # #('dx1000m', 'dz10m',  'Anisotropic') :   ((1.75, 1.875), {'ls' : '--', 'color' : colors[2],    'lw' : 2}, 2, 287), \
    ('dx100m',  'dz10m',  'Anisotropic') : ((6,7),   {'ls' : '--', 'color' : colors[3],    'lw' : 1}, 20, 50), \
    # #('dx100m',  'dz10m',  'Anisotropic') : ((1.5,2),   {'ls' : '--', 'color' : colors[3],    'lw' : 1}, 20, 50), \
    # #('dx50m',   'dz5m',   'Anisotropic_Window_tracers') :   ((2,2.5),   {'ls' : '--', 'color' : colors[4],    'lw' : 1.5}, 40, 25), \
    # # ('dx25m',   'dz2.5m', 'Anisotropic') : ((2,3),   {'ls' : '--', 'color' : colors[5],    'lw' : 1}, 80, 12.5), \
}

data_to_plot = data_iso | data_aniso
outname = 'isotropic_anisotropic'

# data_to_plot = data_aniso
# outname = 'anisotropic'

#data_to_plot = data_iso
#outname = 'isotropic'

# load the data

In [ ]:
data = {}
data_DSD = {}

In [ ]:
%%time
for (dx, dz, sgs), *r in data_to_plot.items():
    simname = dx + ' ' + dz + ' ' + sgs
    print(simname)
    if simname not in data:
        print(datadir[dz][dx][sgs])
        #data[simname] = xrU.load_outdir(dir)
        data[simname], data_DSD[simname] = xrU.load_outdir(datadir[dz][dx][sgs])
        data[simname] = xrU.calc_all(data[simname])
        #data[simname] = xrU.calc_precip_flux(data[simname])
        data[simname] = xrU.convert_units(data[simname])
        #data[simname] = xrU.calc_cloud_base(data[simname], xrU.is_cloudy(data[simname], "rico")) # using rico conditions, because dycoms one cant be used with 1-mom data
        #data[simname] = xrU.calc_cloud_top(data[simname], xrU.is_cloudy(data[simname], "rico"))
        #data[simname] = xrU.calc_zi(data[simname], xrU.zi(data[simname], "dycoms"))
        #data[simname] = xrU.calc_zi(data[simname], xrU.zi(data[simname], "dycoms"))
        #data_DSD[simname] = xrU.calc_all(data_DSD[simname]).pipe(xrU.convert_units)
    
        #force computation of some of the heavier variables - takes time
        data[simname].lwp.load()
        #data[simname].rwp.load()
        #data[simname].zi.load()
        #data[simname].cb_z.load()
        #data[simname].ct_z.load()
        # data[simname].albedo.load()

# TOP-DOWN VIEW

In [ ]:
res_to_plot=[('dx5000m', 'dz50m'), ('dx2000m', 'dz20m'), ('dx100m', 'dz10m'), ('dx50m', 'dz5m'), ('dx25m', 'dz2.5m')]
res_to_plot=[('dx50m', 'dz5m'), ('dx100m', 'dz10m'), ('dx1000m', 'dz10m'), ('dx1000m', 'dz20m'), ('dx2000m', 'dz20m')]
sgs_to_plot=['Isotropic', 'Isotropic_H', 'Isotropic_H100']
sgs_to_plot=['Isotropic', 'Anisotropic']
sgs_to_plot=['Isotropic']
res_to_plot=[('dx50m', 'dz5m'), ('dx300m', 'dz30m'), ('dx2000m', 'dz20m'), ('dx5000m', 'dz50m')]
#sgs_to_plot=['Isotropic', 'Anisotropic', 'Isotropic_Window']
res_to_plot=[('dx50m', 'dz5m')]
sgs_to_plot=['Isotropic_Window', 'Anisotropic_Window', 'Isotropic', 'Anisotropic']
# res_to_plot=[('dx300m', 'dz30m')]
# sgs_to_plot=['Isotropic_Window_tracers_large', 'Anisotropic_Window_tracers']
# sgs_to_plot=['Isotropic_Window_tracers', 'Anisotropic_Window_tracers']

res_to_plot=[('dx5000m', 'dz50m'), ('dx2000m', 'dz20m'), ('dx100m', 'dz10m'), ('dx50m', 'dz5m')]
# sgs_to_plot=['Isotropic_Window', 'Anisotropic_Window','Isotropic', 'Anisotropic','Isotropic_large', 'Anisotropic_large']
sgs_to_plot=['Isotropic', 'Anisotropic']

# res_to_plot=[('dx300m', 'dz30m'), ('dx2000m', 'dz20m')]
# sgs_to_plot=['Anisotropic_Window_tracers']
# res_to_plot=[('dx2000m', 'dz20m')]
# sgs_to_plot=['Isotropic_Window_tracers', 'Isotropic_tracers', 'Anisotropic_Window_tracers', 'Anisotropic_tracers']
# res_to_plot=[('dx50m', 'dz5m')]
# sgs_to_plot=['Anisotropic_Window', 'Anisotropic']
# res_to_plot=[('dx5000m', 'dz50m'), ('dx50m', 'dz5m')]
# sgs_to_plot=['Anisotropic']

plotname = 'IsoAniso'


# Match journal font size standards (usually 8-10 pt)
plt.rcParams.update({
    'font.size': 6,
    'axes.labelsize': 6,
    'axes.titlesize': 6,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'figure.titlesize': 6
})

ncol = max(len(res_to_plot), len(sgs_to_plot))
nrow = min(len(res_to_plot), len(sgs_to_plot))
# fig, ax = plt.subplots(nrow,ncol, figsize=(16.5,6), layout='compressed')
fig, ax = plt.subplots(nrow,ncol, figsize=(7,4.32), layout='compressed')



if len(res_to_plot) > len(sgs_to_plot):
    col_data = res_to_plot
    row_data = sgs_to_plot
    res_in_col = True
else:
    col_data = sgs_to_plot
    row_data = res_to_plot
    res_in_col = False

#for row, sgs_to_plot in enumerate(['Isotropic', 'Anisotropic']):
for row, pr in enumerate(row_data):
    for col, pc in enumerate(col_data):
        try:
            axrc = ax.flatten()[row*ncol+col]
        except:
            axrc = ax
        #if ncol > 1:
        #    axrc = ax[row,col]
        #elif nrow > 1:
        #    axrc = ax[row]
        #else:
        #    axrc=ax
        sgs = pr if res_in_col else pc
        (dx, dz) = pc if res_in_col else pr
        simname = dx + ' ' + dz + ' ' + sgs
        print(data[simname].datadir)
        t_av_end = data_to_plot[(dx, dz, sgs)][0][1] # end time of the averaging period
        #t_av_end = 2.25
        _data = data[simname]        
        
        cmap='Blues_r'
        vmin, vmax = None, None
        
        #coarsen to 2km
        #scale = data_to_plot[(dx, dz, sgs)][2]
        #_data = data[simname].coarsen(x=scale, y=scale, boundary="trim").mean()        

        # plot only part of the domain
        #_data = _data.sel(x=slice(0,25000), y=slice(0,25000))

        # -- plotting scalars
        # - albedo
        # var = _data.albedo.sel(t=t_av_end, method='nearest')
        # varname = 'albedo'
        # varlabel = 'albedo [1]'
        # cmap='Blues_r'
        # vmin, vmax = 0, 1
        # - LWP
        var = _data.lwp.sel(t=t_av_end, method='nearest')
        varname = 'lwp'
        varlabel = 'LWP [g/m$^2$]'
        # - tr_col@750m
        #var = _data.tr_col.sel(t=t_av_end, method='nearest').sel(z=750, method='nearest')
        #varname = 'tr_col@750m'
        #varlabel = varname
        # - w@200m like in Wang 2009
        # cmap = mpl.colors.ListedColormap(['blue', 'turquoise', 'white', 'orange', 'red'], name="w_cmap")
        # var = _data.w.sel(t=t_av_end, method='nearest').sel(z=200, method='nearest')
        # varname = 'w@200m'      
        # varlabel = 'w@200m'  
        
        #var_scaled = var / var.mean()
        im = var.plot(x="x", cmap=cmap, ax=axrc, add_colorbar=False, vmin=vmin, vmax=vmax, rasterized=True)
        #im = var.plot(x="x", cmap='bwr', ax=axrc, add_colorbar=False)
        if col == ncol-1:
            fig.colorbar(im, ax=axrc).set_label(varlabel)
        else:
            fig.colorbar(im, ax=axrc)

        '''
        # add squares (for initial column tracers positions)
        x=4e3 + _data.di # shift by one cell size due to uwlcm error in tr_col init
        size=2e3
        while x < _data.x[-1]:
            y=4e3 + _data.dj
            while y < _data.y[-1]:
                square = mpl.patches.Rectangle((x, y), size, size, edgecolor='black', facecolor='none', ls=':')
                axrc.add_patch(square)
                y+=6e3
            x+=6e3
            
        # add squares (for column tracers positions shifted by mean horizontal velocity)
        # TODO: what if it goes out of bounds? apply periodic bcond
        x=4e3 + _data.di
        y=4e3 + _data.dj
        tstart = math.floor(t_av_end) # tr_col is reinitialized every hour, but output at full hours is done before reinitialization
        if tstart == t_av_end:
            tstart -= 1
        tend = var.t.values # t_av_end might not be an output time, so find the closest real output time
        # trapezoidal integration of position
        for t in _data.sel(t=slice(tstart, tend)).t.values:        
            u_mean = _data.u.sel(t=t, method='nearest').sel(z=750, method='nearest').mean(["x","y"]).values
            v_mean = _data.v.sel(t=t, method='nearest').sel(z=750, method='nearest').mean(["x","y"]).values
            h = _data.outfreq * _data.dt
            if t == tstart or t == tend:
                h /= 2.
            x += h * u_mean
            y += h * v_mean
            print(simname, t, u_mean, v_mean)
        xs = x
        while xs < _data.x[-1]:
            ys = y
            while ys < _data.y[-1]:
                square = mpl.patches.Rectangle((xs, ys), size, size, edgecolor='black', facecolor='none')
                axrc.add_patch(square)
                ys+=6e3
            xs+=6e3
            '''
            



        #plotting vector field
        #alt = 800 # altitude of cross section [m]
        #u = _data.u.sel(t=t_av_end, method='nearest').sel(z=400, method='nearest')
        #v = _data.u.sel(t=t_av_end, method='nearest').sel(z=400, method='nearest')
        #up = u - u.mean(["x","y"])
        #vp = v - v.mean(["x","y"])
        ##print(_data.x, _data.y, u, v)
        ##print(_data.x.to_numpy(), _data.y.to_numpy(), u.to_numpy(), v.to_numpy())
        #speed = np.sqrt(up.to_numpy()**2 + vp.to_numpy()**2)
        #print(speed.min(), speed.max())
        #lw = 5*(speed / speed.max())**2
        #strm = axrc.streamplot(_data.x.to_numpy(), _data.y.to_numpy(), up.to_numpy(), vp.to_numpy(), color=speed, cmap='autumn', linewidth=lw)
        #fig.colorbar(strm.lines, ax=axrc)
        ##strm = axrc.quiver(_data.x.to_numpy(), _data.y.to_numpy(), up.to_numpy(), vp.to_numpy())#, color=speed, cmap='autumn', linewidth=lw)
        #varname = 'uv400m'
        
        # labels etc.
        axrc.set_xticks([])
        axrc.set_yticks([])
        axrc.set_title('')
        axrc.set_ylabel('')
        axrc.set_xlabel('')
        axrc.set_box_aspect(1)
        #if row == 0:
        if row == nrow-1:
            axrc.set_xlabel(str(data_to_plot[(dx, dz, sgs)][3]) + ' km')               
        # axrc.set_ylabel(str(data_to_plot[(dx, dz, sgs)][3]) + ' km')               
        #if col == 0:
        #axrc.set_ylabel(sgs)
        #axrc.set_ylabel(sgs_nicename[sgs])
        if row == 0:
            axrc.set_title(resolution_nicename[dx] + ' ' + resolution_nicename[dz])# + ' t=' + str(t_av_end) + 'h')   
        # axrc.set_title(sgs_nicename[sgs] + ' @' + str(t_av_end) + 'h')
        #_data.w.sel(z=300, method='nearest').sel(t=t_av_end, method='nearest').plot.contour(x="x", add_colorbar=True, ax=axrc)#, vmin=-0.6, vmax=0.6)
        if col == 0:
            # Add vertical row label further to the left of the y-label
            axrc.annotate(
                'Isotropic Smagorinsky' if row == 0 else 'Anisotropic Smagorinsky',
                xy=(-0.05, 0.5),          # x < 0 shifts left; y = 0.5 centers vertically
                xycoords='axes fraction',
                rotation=90,               # Rotates text vertically (270 for top-to-bottom)
                ha='right', 
                va='center',
                fontsize=6,
                fontweight='bold'
            )

            
        # subplot labels
        for i, axc in enumerate(ax.flat):
            axc.text(
                0.05, 0.85, 
                f"({string.ascii_lowercase[i]})", 
                transform=axc.transAxes,
                fontsize=8,
                color='white'
            )

# plt.tight_layout()
plt.savefig(figoutdir+"/topdown/"+str(plotname)+"_"+str(varname)+"_topdown.png", dpi=300, bbox_inches='tight', pad_inches=0.05)
plt.savefig(figoutdir+"/topdown/"+str(plotname)+"_"+str(varname)+"_topdown.pdf", dpi=300, bbox_inches='tight', pad_inches=0.05)

# Reset to typical font sizes
plt.rcParams.update({
    'font.size': 7,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.titlesize': 8
})

In [ ]:
# lwp coarsened
scale = int(2000 / 50) # to 2km from 50m
data[sgs_to_plot].lwp.sel(t=24000, method='nearest').coarsen(x=scale, y=scale, boundary="trim").mean().plot(x="x", cmap='Blues_r')
plt.savefig(figoutdir+"/topdown/"+str(dx_to_plot)+"_"+str(dz_to_plot)+"_coarsened_to_dx2000m_"+str(sgs_to_plot)+"_"+"lwp"+"_topdown.png", dpi=300)

# TIME SERIES

In [ ]:
%%time

# Function to select square areas with spacing. Useful for analyzing tr_col (tracers initialized in columns)
def select_square_areas_with_spacing(ds, square_size, spacing):
    selected_areas = []
    for i in np.arange(spacing - square_size, ds.attrs['X'], spacing): # [km]->[m] in dims
        for j in np.arange(spacing - square_size, ds.attrs['Y'], spacing):
            off = [1.5*ds.di, 1.5*ds.dj] #offset between (x,y) positions in the DataSet and (blitz::i-1)*dx, which is used to init tr_col
            selected_area = ds.sel(x=slice(i+off[0], i+off[0] + square_size-0.01), y=slice(j+off[1], j+off[1] + square_size-0.01)) #-0.01 because slice is inclusive on both ends
            selected_areas.append(selected_area)
    return selected_areas

plot_parameters = {
    #'tr_inv' : dict(reduce='3D_mean'),
    #'tr_col' : dict(reduce='3D_mean'), # can take long time
    #'tr_col_sdev@750m' : dict(reduce='2D_sdev'), 
    #'zi': dict(reduce='2D_mean'), # NOTE: it may be different if we first calculate an average profile and later calc zi
    'lwp': dict(reduce='2D_mean'),
    # 'albedo': dict(reduce='2D_mean'),
    #'rwp': dict(reduce='2D_mean'),
    #'cb_z': dict(reduce='2D_mean'), # 2D_min
    #'ct_z': dict(reduce='2D_mean'), # 2D_max
    #'prflux_s': dict(reduce='2D_mean'),
    #'prflux_cb': dict(reduce='2D_mean'),
    #'acc_precip_s': dict(reduce='none'),
    #'acc_precip_cb': dict(reduce='2D_sum'),
    #'latent surface flux': dict(reduce='2D_mean'),
    #'sensible surface flux': dict(reduce='2D_mean'),
    #'w_var_ct': dict(reduce='2D_mean'),
    'cell_scale': dict(reduce='none'), # approximate scale of convective cells: 2/3 of variance of lwp is in scales smaller than cell_scale (de roode 2004)
}

for varname, params in plot_parameters.items():
    fig, ax = plt.subplots(1, 2, figsize=(3.5,2.16), layout='compressed', sharey=True)    
    for (dx, dz, sgs), ((ts, te), p, c, d) in data_to_plot.items():
        axc = ax[0] if d <= 50 else ax[1] # LES simulations (domain <= 50km) in first plot, convection-permitting in second        
        simname = dx + ' ' + dz + ' ' + sgs
        # some vars need to be calculated (TODO: move it to calculate.py?)
        if varname == 'prflux_s':
            _data=data[simname]['prflux'].sel(z=0, method='nearest')
            _data.attrs["long_name"]="surface precipitation flux"
            _data.attrs["units"]="W/m$^2$"
        elif varname == 'prflux_cb':
            _data=data[simname]['prflux'].sel(z=data[simname]['cb_z'].min(["x","y"]), method='nearest')
            #_data=data[simname]['prflux'].sel(z=700, method='nearest') # hardcoded 700m
            _data.attrs["long_name"]="cloud base precipitation flux"
            _data.attrs["units"]="W/m$^2$"
        elif varname == 'acc_precip_s':
            _data=data[simname]['puddle_liquid_volume']/data[simname]['surf_area']*1e3
            _data.attrs["long_name"]="accumulated surface precipitation"
            _data.attrs["units"]="mm"
        elif varname == 'acc_precip_cb':
            #_data=data[simname]['prflux'].sel(z=data[simname]['cb_z'].min(["x","y"]), method='nearest') \
            #    / xrU.L_evap * data[simname].outfreq * data[simname].dt
            _data=data[simname]['prflux'].sel(z=700, method='nearest') \
                / xrU.L_evap * data[simname].outfreq * data[simname].dt
            _data=_data.cumsum()
            _data.attrs["long_name"]="accumulated cloud base precipitation"
            _data.attrs["units"]="mm"
        elif(varname == 'w_var_ct'):
            ct_mean = data[simname].ct_z.mean(["x","y"])
            _data=pow(data[simname]['w'].sel(z=ct_mean, method='nearest'),2)
            _data.attrs["long_name"] = "variance of w at cloud top"
            _data.attrs["units"] = "m$^2$/s$^2$"
        elif(varname == 'tr_col'): # time evolution of tr_col in columns that initially have tr_col=1 (whats missing must have mixed); TODO: shift this column by mean horizontal velocity as done in topdown plots 
            _data=data[simname].sel(z=slice(300,850))
            square_size = 2000 #[m]
            spacing = 6000 #[m]
            squares = select_square_areas_with_spacing(_data, square_size, spacing)
            #merge the squares
            _data=xr.concat(squares[:min(10,len(squares))], dim='x').tr_col.chunk({"t":1}) # only up to 10 squares. otherwise it is very slow
        elif(varname == 'tr_col_sdev@750m'): # standard deviation of tr_col at 750meters
            _data=data[simname]['tr_col'].sel(z=750, method='nearest')
        elif(varname == 'tr_inv'): # time evolution of tr_inv below 850m (TODO: change to below inversion?) (initially its 1 above 850m and 0 below)
            _data=data[simname].tr_inv.where(data[simname].z<data[simname].zi)#data[simname].zi.mean(["x","y"])))
        elif(varname == 'cell_scale'): # based on lwp, because we have lwp from satellites; other variables (e.g. q_t, u, v) give similar scales (also see Zhou et al. 2018);
            #see the energy_spectra cell for more details
            _data = data[simname]['lwp']
            # _data = data[simname]['albedo']
            _data = _data.drop_sel(x=_data.coords['x'].values[0], y=_data.coords['y'].values[0])             
            # Create a DataArray with the same time dimension as E
            cell_scale = np.zeros(len(_data.coords['t'].values))
            for t in range(cell_scale.shape[0]):
                E, K, kx, ky = xrU.calc_spectrum2d(_data.sel(t=_data.coords['t'].values[t]).to_numpy(), [0,1], data[simname].di, 2) 
                E_int, lmbd = xrU.spectrum2d_rad_mean_integration(K, E, kx, ky)
                xrU.interpolate_nan(E_int, lmbd)
                E_int /= lmbd # multiply by K as in de roode 2004
                E_int = xrU.smooth_spectrum(E_int)
                max_index = np.argmin(-E_int)
                cell_scale[t] = lmbd[max_index] / 1e3 # in [km]
                # cumsum = np.cumsum(E_int)
                # cell_scale[t] = lmbd[np.argmin(np.abs(cumsum - 2./3*cumsum[-1]))]
            # Convert the list of integrated values to a DataArray
            _data = xr.DataArray(
                np.array(cell_scale),
                dims=["t"],
                coords={"t": _data.coords['t'].values},
                name="cell_scale"
            )
            _data.t.attrs["units"] = "h"
            _data.t.attrs["long_name"] = "time"
            _data.attrs["long_name"] = "scale of cloud cells"
            _data.attrs["units"] = "km"
        else:
            _data=data[simname][varname]
                        
        if params['reduce']=='3D_mean':
            res=_data.mean(["x","y","z"], keep_attrs=True)
        elif params['reduce']=='3D_sum':
            res=_data.sum(["x","y","z"], keep_attrs=True)
        elif params['reduce']=='2D_mean':
            res=_data.mean(["x","y"], keep_attrs=True)
        elif params['reduce']=='2D_sum':
            res=_data.sum(["x","y"], keep_attrs=True)
        elif params['reduce']=='2D_min':
            res=_data.min(["x","y"], keep_attrs=True)
        elif params['reduce']=='2D_max':
            res=_data.max(["x","y"], keep_attrs=True)
        elif params['reduce']=='2D_sdev':
            res=_data.std(["x","y"], keep_attrs=True)
        elif params['reduce']=='none':
            res=_data
        #res.to_dataset(name=varname).to_netcdf(ncoutdir+"/se`ries/res"+str(resolution_to_plot)+"_"+str(aerosol_to_plot)+"_series.nc", group=simname, mode='a', engine='h5netcdf')
        res = res.where(res.t <= te)
        #print(data[simname].lwp)
        label = resolution_nicename[dx] + ' ' + resolution_nicename[dz] + ' ' + sgs_nicename[sgs]
        
        res.plot(xlim=(0, None), label=label, ax=axc, **p)
        #res.plot(xlim=(14, 24), label=simname, **p)
        #res.plot(xlim=(0, 2))

    
    # subplot labels
    for i, axc in enumerate(ax.flat):
        axc.text(
            0.05, 0.9, 
            f"({string.ascii_lowercase[i]})", 
            transform=axc.transAxes,
            fontsize=8
        )
    
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=2, fontsize=6)
    #plt.xlabel('t [h]')
    #plt.ylabel(params['nicename']+' '+params['units'])
    ax[1].set_ylabel('')
    plt.title('')
    plt.savefig(figoutdir+"/series/"+str(outname)+"_"+str(varname)+"_series.png", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.savefig(figoutdir+"/series/"+str(outname)+"_"+str(varname)+"_series.pdf", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.show()
    plt.clf()
#print(_data[0].tr_col)
#print(_data[0].sel(t=0).tr_col.mean(["x", "y"]).values)

# PROFILES 

In [ ]:
%%time

plot_parameters = {
    #'tr_inv' : dict(),
    # 'u': dict(), #dict(nicename='$v$', units='[m/s]'),
    # 'v': dict(), 
    # 'rv': dict(),
    # 'rc': dict(),
    # 'rr': dict(),
    # 'th': dict(),
    ##'temp': dict(),
    # 'RH': dict(),
    #'radiative_flux': dict(),
    #'sgs_rv_flux': dict(),
    #'sgs_th_flux': dict(),
    'u_var': dict(),
    'v_var': dict(),
    'w_var': dict(),
    'w_3rd': dict(),
    'k_ma[0]': dict(),
    'k_ma[1]': dict(),
    #'tke': dict(),
    
    #'rv': dict(nicename='$r_v$', units='[g/kg]'),
    #'rc': dict(nicename='$r_c$', units='[g/kg]'),
    #'rr': dict(nicename='$r_r$', units='[g/kg]'),
    #'th': dict(nicename='$\\theta$', units='[K]'),
    ##'temp': dict(nicename='T', units='[K]'),
    #'RH': dict(nicename='RH', units='[1]'),
    #'radiative_flux': dict(nicename='radiative flux', units='[W/m$^2$]'),
    #'sgs_rv_flux': dict(nicename='sgs $r_v$ flux', units='[W/m$^2$]'),
    #'sgs_th_flux': dict(nicename='sgs $\\theta$ flux', units='[W/m$^2$]'),
    #'w_var': dict(nicename='varianve of $w$', units='[$m^2/s^2$]'),
    #'w_3rd': dict(nicename='3rd moment of $w$', units='[$m^3/s^3$]'),
    #'k_ma[0]': dict(nicename='k_ma horizontal', units='[?]'),
    #'k_ma[1]': dict(nicename='k_ma vertical', units='[?]'),
#    'tke': dict(nicename='tke', units='[?]'),
}

for varname, params in plot_parameters.items():
    fig, ax = plt.subplots(1, 2, figsize=(3.5,2.16), layout='compressed', sharey=True)    
    for (dx, dz, sgs), ((ts, te), p, c, d) in data_to_plot.items():
        axc = ax[0] if d <= 50 else ax[1] # LES simulations (domain <= 50km) in first plot, convection-permitting in second        
        simname = dx + ' ' + dz + ' ' + sgs
        #print(simname)
        #print(data[simname])
        if (varname == 'sgs_rv_flux' or varname == 'sgs_th_flux'):
            if 'sgs_scheme' not in data[simname].attrs:
                _data=xr.zeros_like(data[simname]['rv'])
            else: # missing '-' sign in UWLCM output
                _data=-data[simname][varname]
        elif(varname == 'u_var'):
            _data=pow(data[simname]['u'] - data[simname]['u'].mean(["x","y"]),2)
            _data.attrs["standard_name"] = r"$<u'^2>$"
            _data.attrs["long_name"] = "variance of u"
            _data.attrs["units"] = "m$^2$/s$^2$"
        elif(varname == 'v_var'):
            _data=pow(data[simname]['v'] - data[simname]['v'].mean(["x","y"]),2)
            _data.attrs["standard_name"] = r"$<v'^2>$"
            _data.attrs["long_name"] = "variance of v"
            _data.attrs["units"] = "m$^2$/s$^2$"
        elif(varname == 'w_var'):
            _data=pow(data[simname]['w'],2)
            _data.attrs["standard_name"] = r"$<w'^2>$"
            _data.attrs["long_name"] = "variance of w"
            _data.attrs["units"] = "m$^2$/s$^2$"
        elif(varname == 'w_3rd'):
            _data=pow(data[simname]['w'],3)
            _data.attrs["standard_name"] = r"$<w'^3>$"
            _data.attrs["long_name"] = "3rd moment of w"
            _data.attrs["units"] = "m$^3$/s$^3$"
        elif((varname == 'k_ma[0]' or varname == 'k_ma[1]') and 'k_m' in data[simname]): # isotropic SMG stores k_m (single value)
            _data=data[simname]['k_m']
        elif(varname == 'u'):
            _data=data[simname]['u'] + data[simname].u_mean
        elif(varname == 'v'):
            _data=data[simname]['v'] + data[simname].v_mean
        else:
            if varname not in data[simname]:
                _data=xr.zeros_like(data[simname]['rv'])
            else:
                _data=data[simname][varname]

        label = resolution_nicename[dx] + ' ' + resolution_nicename[dz] + ' ' + sgs_nicename[sgs]

        #_data = _data.where(data[simname].lwp>5) # filter non-cloudy columns
        # Convert z to km and set units attribute
        plot_data = _data.assign_coords(z=_data.z / 1000)
        plot_data.z.attrs["units"] = "km"
        
        # Plot with updated ylim in km
        plot_data.where(plot_data.t > ts).where(plot_data.t <= te).mean(
            ["x", "y", "t"], keep_attrs=True
        ).plot(y="z", ylim=(0, 1.5), label=label, ax=axc, **p)

        var_name = (
            _data.attrs.get("standard_name")
            or _data.attrs.get("long_name")
            or _data.name
        )

        units = (_data.attrs.get("units") or '1')

        axc.set_xlabel(var_name + ' [' + units + ']')

    # subplot labels
    for i, axc in enumerate(ax.flat):
        axc.text(
            0.85, 0.9, 
            f"({string.ascii_lowercase[i]})", 
            transform=axc.transAxes,
            fontsize=8
        )
    
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=2, fontsize=6)
    ax[1].set_ylabel('')
    #plt.ylabel('z [m]')
    #plt.xlabel(params['nicename']+' '+params['units'])
    #plt.title('mean profile between ' + str(timestart) +'h and '+str(timeend)+'h')
    plt.savefig(figoutdir+"/profiles/"+str(outname)+"_"+str(varname)+"_profiles.png", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.savefig(figoutdir+"/profiles/"+str(outname)+"_"+str(varname)+"_profiles.pdf", dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.show()
    plt.clf()


# LWP histogram #

In [ ]:
%%time

#bins = np.arange(5, 500, 20) # NOTE: clear-sky (<5 g/m2) excluded
bins = np.arange(5./100., 500./100., 20./100) # for lwp divided by mean

coarsen = False # should lwp be coarsened to 2km (5km for dx=5km) before plotting histogram

#for simname in data:
fig, ax = plt.subplots(1, 2, figsize=(3.5,2.16), layout='compressed', sharey=True)    
for (dx, dz, sgs), (t, p, c, d) in data_to_plot.items():
    axc = ax[0] if d <= 50 else ax[1] # LES simulations (domain <= 50km) in first plot, convection-permitting in second        
    simname = dx + ' ' + dz + ' ' + sgs
    #(ts, te) = averaging_period[simname]
    _data = data[simname]
    _data = _data.where(_data.t>t[0]).where(_data.t<=t[1])
    if coarsen:
        _data = _data.coarsen(x=c, y=c, boundary="trim").mean() # coarsen
    _data /= _data.mean() # scale by the mean
    #print(_data.lwp.values)
    #plt.hist(_data.lwp)
    label = resolution_nicename[dx] + ' ' + resolution_nicename[dz] + ' ' + sgs_nicename[sgs]    
    xr.plot.hist(_data.lwp, label=label, bins=bins, histtype='step', density=True, ax=axc, **p)
    #xr.plot.hist(_data.lwp / _data.lwp.max(), label=simname, bins=100, histtype='step', density=True)

    
# subplot labels
for i, axc in enumerate(ax.flat):
    axc.text(
        0.85, 0.9, 
        f"({string.ascii_lowercase[i]})", 
        transform=axc.transAxes,
        fontsize=8
    )

#plt.yscale('log')
fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=2, fontsize=6)
ax[0].set_ylabel('PDF')
#plt.xlabel('lwp [g/m$^2$]')
for axi in ax:
    axi.set_xlabel('LWP / <LWP>')
    axi.set_xlim(-20./100,500./100)

histname = "_lwp_scaled_histogram" if not coarsen else "_lwp_scaled_CoarsenedTo2km_histogram"
plt.savefig(figoutdir+"/histograms/"+str(outname)+histname+".png", dpi=300, bbox_inches='tight', pad_inches=0.05)
plt.savefig(figoutdir+"/histograms/"+str(outname)+histname+".pdf", dpi=300, bbox_inches='tight', pad_inches=0.05)
plt.show()

# RWP vs LWP #

In [ ]:
plt.rcParams.update({
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.titlesize': 9
})

lwp = {}
lwp['Isotropic'] = []
lwp['Anisotropic'] = []
rwp = {}
rwp['Isotropic'] = []
rwp['Anisotropic'] = []

# black for 10x grid ratio, red for 100x
annotation_color = {\
    "dx5000m" : "red" ,\
    "dx2000m" : "red" ,\
    "dx1000m" : "red" ,\
    "dx100m" : "black" ,\
    "dx50m" : "black" ,\
    "dx25m" : "black" ,\
}

plt.figsize=(3.5,2.16)

for (dx, dz, sgs), ((ts, te), p, c, d) in data_to_plot.items():
    simname = dx + ' ' + dz + ' ' + sgs
    lwp[sgs].append(data[simname].lwp.sel(t=te, method='nearest').mean(["x","y"], keep_attrs=True))
    rwp[sgs].append(data[simname].rwp.sel(t=te, method='nearest').mean(["x","y"], keep_attrs=True))
    if sgs == "Anisotropic" and dx == "dx100m": # this annotation needs to be moved to be visible
        plt.annotate(resolution_nicename[dx], (lwp[sgs][-1].values, rwp[sgs][-1].values), xytext=(0,-1), textcoords='offset fontsize', color=annotation_color[dx])
    elif sgs == "Isotropic" and dx == "dx100m": # ditto
        plt.annotate(resolution_nicename[dx], (lwp[sgs][-1].values, rwp[sgs][-1].values), xytext=(-1.5,0.5), textcoords='offset fontsize', color=annotation_color[dx])
    elif sgs == "Anisotropic" and dx == "dx2000m": # ditto
        plt.annotate(resolution_nicename[dx], (lwp[sgs][-1].values, rwp[sgs][-1].values), xytext=(-6,0.5), textcoords='offset fontsize', color=annotation_color[dx])
    else:
        plt.annotate(resolution_nicename[dx], (lwp[sgs][-1].values, rwp[sgs][-1].values), xytext=(0,0.5), textcoords='offset fontsize', color=annotation_color[dx])

plt.plot(lwp['Isotropic'], rwp['Isotropic'], marker='*', markersize=7, linestyle=':', label='Isotropic')
plt.plot(lwp['Anisotropic'], rwp['Anisotropic'], marker='*', markersize=7, linestyle=':', label='Anisotropic')

#for lwp, rwp in zip(lwp['Isotropic'], rwp['Isotropic']):
#    plt.annotate('asd', (lwp.values, rwp.values))


plt.xlabel('LWP [g/m$^2$]')
plt.ylabel('RWP [g/m$^2$]')
plt.legend()
plt.savefig(figoutdir+"rwp_vs_lwp/UWLCM_rwp_vs_lwp.png", dpi=300, bbox_inches='tight')
plt.savefig(figoutdir+"rwp_vs_lwp/UWLCM_rwp_vs_lwp.pdf", dpi=300, bbox_inches='tight')

# Energy spectra 

In [ ]:

plot_parameters = {
    #'u': {"exp" : 2, "ylabel": "E$_u$ [$m^3 s^{-2}$]", "height_range": (650, 850)}, # exp=2 mean power spectrum # (650 850)
    #'v': {"exp" : 2, "ylabel": "E$_v$ [$m^3 s^{-2}$]", "height_range": (650, 850), "t" : (20.5,23.5)}, 
    # 'w': {"exp" : 2, "ylabel": "E$_w$ [$m^3 s^{-2}$]", "height_range": (650, 850)}, # "t" : (23.5,23.5)}, 
    # 'lwp': {"exp" : 2, "ylabel": "LWP [$g^2/m^3$]", "height_range": None}, 
    'lwp': {"exp" : 2, "ylabel": r"$k E_{LWP}\, [\mathrm{g}^2/\mathrm{m}^4$]", "height_range": None}, 
    # 'albedo': {"exp" : 2, "ylabel": "$k E_{albedo}$ [1/m]", "height_range": None}, #, "t" : (3.5,3.5)}, 
    # 'rt': {"exp" : 2, "ylabel": "q$_t$", "height_range": (650, 850)},# "t" : (23.5,23.5)}, 
    #'lwp@15h': {"varname" : "lwp", "exp" : 2, "ylabel": "LWP [$g^2/m^3$]", "height_range": None, "t" : (15,15)}, 
    #'lwp@10h': {"varname" : "lwp", "exp" : 2, "ylabel": "LWP [$g^2/m^3$]", "height_range": None, "t" : (10,10)}, 
    #'tr_col' : {"exp" : 1, "ylabel": "A$_t$ [m]"}, # exp=1 means fourier amplitude spectrum
    #'tr_col@0h' : {"varname" : "tr_col", "t" : (0,0), "exp" : 1, "ylabel": "A$_t$ [m]"}, # exp=1 means fourier amplitude spectrum
    #'tr_col@15min' : {"varname" : "tr_col", "t" : (0.25,0.25), "exp" : 1, "ylabel": "A$_t$ [m]"}, # exp=1 means fourier amplitude spectrum
    #'tr_col@1h' : {"varname" : "tr_col", "t" : (1,1), "exp" : 1, "ylabel": "A$_t$ [m]"},
    #'tr_col@2h' : {"varname" : "tr_col", "t" : (2,2), "exp" : 1, "ylabel": "A$_t$ [m]"},
    #'tr_col@23.5h' : {"varname" : "tr_col", "t" : (23.5,23.5), "exp" : 1, "ylabel": "A$_t$ [m]"},
    #'tr_col@24h' : {"varname" : "tr_col", "t" : (24,24), "exp" : 1, "ylabel": "A$_t$ [m]"},
}

E_lst = {} # we store values for u v and w in order to plot anisotropicity parameter R=E(u)+E(v)/2*E(w)

for varname, params in plot_parameters.items():
    E_lst[varname] = {}
    _varname = varname if "varname" not in params else params["varname"]
    exp = params["exp"]
    height_range = params["height_range"]
    fig, ax = plt.subplots(2, 2, figsize=(3.5,3.5), layout='compressed', sharey=False)    
    for it, ((dx, dz, sgs), ((_ts, _te), p, c, d)) in enumerate(data_to_plot.items()):
        # axc = ax.flatten()[it%4]
        if dx == 'dx5000m':
            axc = ax[1,1]
        elif dx == 'dx2000m':
            axc = ax[1,0]
        elif dx == 'dx100m':
            axc = ax[0,1]
        else:
            axc = ax[0,0]
        simname = dx + ' ' + dz + ' ' + sgs
        (ts, te) = params["t"] if "t" in params else (_ts, _te)
        # print(it, simname, _ts, _te, p, c, d)
        if te > _te:
            print("Specified te exceeds simulation time. Skipping.")
            continue
        _data = data[simname][_varname]

        # if needed, select height levels
        if(height_range != None):
            _data = _data.sel(z = slice(height_range[0], height_range[1])) # for a range of heights
            #_data = _data.sel(z=height, method='nearest') # for single height
            hname = "_between_"+str(height_range[0])+"m_and_"+str(height_range[1])+"m"
        else:
            hname = ""
        
        _data = _data.sel(t = slice(ts,te))
        # drop edge cells since for periodic bcond they are the same as the opposite edge
        _data = _data.drop_sel(x=_data.coords['x'].values[0], y=_data.coords['y'].values[0]) 

        # !!! Normalize, to get spectrum of fluctuations relative to the mean value, not of the mean!
        # _data = _data / _data.mean(["x","y"]) 

        # --- first method: 2d fft and integrate radially  ---
        E, K, kx, ky = xrU.calc_spectrum2d(_data, [1,2], data[simname].di, exp) 
        # print(kx[len(kx)//2], ky[len(ky)//2])
        # print(E.shape, K.shape)
        # print(E[0][K==0])
        # -- angular integration using spline interpolation --
        # E_int, lmbd = xrU.spectrum2d_rad_spline_integration(kx, ky, E[0,:,:],51) # spectrum2d_rad_spline_integration doesnt work yet with multiple time and height
        # E_int /= lmbd # multiply by K as in de roode 2004
        # plt.loglog(lmbd[E_int!=0], E_int[E_int!=0], label= f'{simname} {varname} spline', **p) # E_int=0 means nonexisting wavenumber
        # -- angular integration using averaging with equal weights --
        E_int, lmbd = xrU.spectrum2d_rad_mean_integration(K, E, kx, ky, 101)        
        # print(np.log(lmbd))
        # NaN values mean that there are no points at these k. But having equal dlog(k) is nice, so we interpolate these values
        xrU.interpolate_nan(E_int, lmbd)
        E_int /= lmbd # multiply by K as in de roode 2004, so that energy is propotional to area under curve for logscale x
        lmbd /= 1e3 # to [km], only for plotting. needs to be done after /=lmbd, because E_int/lmbd is in g2/m4
        #find characteristic length scale, lengths shorter than it have 2/3 of the variance (de roode 2004)
        cumsum = np.cumsum(E_int)
        length_scale = lmbd[np.argmin(np.abs(cumsum - 2./3*cumsum[-1]))]        
        #plt.stairs(S_int, lmbd_bins, label='radially integrated with averaging')
        # plot unsmoothed data
        # plt.plot(lmbd, E_int, label= f'{simname} {varname} @{te:.1f}h length scale {length_scale:.2f}', **p, alpha=0.3) # E_int=0 means nonexisting wavenumber
        #Smooth the spectrum to find maximum position and how distinct it is
        #E_smooth = xrU.smooth_spectrum(E_int)
        E_smooth = xrU.smooth_spectrum(E_int)
        max_index = np.argmin(-E_smooth)
        length_scale = lmbd[max_index]
        # asses steepnes of the maximum using second derivative
        # dy_dx = np.gradient(E_smooth, np.log10(lmbd))
        # d2y_dx2 = np.gradient(dy_dx, np.log10(lmbd))        
        # slope_at_max = dy_dx[max_index]
        # curvature_at_max = d2y_dx2[max_index]
        # print(f'slope and curvature in log(lambda): {slope_at_max :.2e}, {curvature_at_max: .2e}')
        # dy_dx = np.gradient(E_smooth, lmbd)
        # d2y_dx2 = np.gradient(dy_dx, lmbd)
        # slope_at_max = dy_dx[max_index]
        # curvature_at_max = d2y_dx2[max_index]
        # print(f'slope and curvature in lambda: {slope_at_max :.2e}, {curvature_at_max: .2e}')
        # Another method - estimate how pronounced is the peak by calculating width at 50%
        # lvl = 0.6 * E_smooth[max_index]
        # plt.axhline(lvl, color='black', linestyle=':', linewidth=1)
        # indices = np.where(E_smooth >= lvl)[0]
        #print(lvl, indices)
        # if len(indices) >= 2: 
        #     diff = np.diff(indices)
        #     # TODO: doesnt work
        #     diff_max_index = max_index - indices[0]# - np.sum(diff-1) #position of max index in the diff array
        #     print(diff, diff[diff_max_index:], diff[:diff_max_index])
        #     if not np.all(diff == 1): # not all indices are consecutive - some local minimum inbetween! Find width of the maximum peak (ie of the consecutive range)
        #         indmax = np.argmax(diff[diff_max_index:]>1)
        #         if diff[indmax]==1: #all censectuvie ond this side of the maximum
        #             indmax = len(diff[diff_max_index:]) 
        #         indmax = indmax + max_index
        #         indmin = np.argmax(np.flip(diff[:diff_max_index]>1))                
        #         if diff[indmin]==1: #all censectuvie ond this side of the maximum
        #             indmin = len(diff[:diff_max_index]) 
        #         indmin = max_index - indmin
        #         print("WARNING: Width calculation cuts through some local minimum!")
        #     else:
        #         indmin, indmax = indices[-1], indices[0]
        #     print(f'width between {lmbd[E_int!=0][indmin] : .2e} and {lmbd[E_int!=0][indmax] : .2e}')
        #     width = lmbd[E_int!=0][indmin] - lmbd[E_int!=0][indmax]
        #     print(width)
        # plt.loglog(lmbd[E_smooth!=0], E_smooth[E_smooth!=0], label= f'{simname} {varname} smoothed/', **p)
        #plt.loglog(lmbd[E_int!=0], E_smooth, label= f'{simname} {varname} smoothed length scale {length_scale:.2f}', **p)
        label = resolution_nicename[dx] + ' ' + resolution_nicename[dz] + ' ' + sgs_nicename[sgs]            
        # axc.plot(lmbd[E_int!=0], E_smooth, label= f'{simname} {varname} @{te:.1f}h smoothed length scale {length_scale:.2f}', **p)
        axc.plot(lmbd[E_int!=0], E_smooth, label=label, **p)
        axc.plot(lmbd[E_int!=0], E_int, label='', alpha=0.5, **p)
        axc.set_xscale('log')
        # plt.yscale('log')
        # plt.gca().invert_xaxis()
        # plt.legend()
        # plt.show()
      
        # second method: 1D fft along x and y and average
        # E, K, lmbd = xrU.calc_spectrum(_data, [1,2], data[simname].di, exp)
        # E_avg, lmbd = xrU.spectrum_average(E, K, lmbd)
        # # get S*k as in de roode 2004
        # # _E[0] *= _K[0][:, np.newaxis]
        # # _E[1] *= _K[1][:, np.newaxis]                    
        # plt.loglog(lmbd, E_avg, 'o' , label=simname+"_"+varname, **p)
        
        #E_lst[varname][simname]=(lmbd, E_avg)
        E_lst[varname][simname]=(lmbd, E_int)
        
        # phase spectrum, along x for now; The goal is to find how far the tr_col columns have moved since initialization, 
        # by looking at phases of the 6km wavelength, as these columns are 6km apart; Stopped working on this because it is easier (and sufficiently accurate?)
        # to move the columns with mean horizontal velocity.
        '''
        Ex = (np.angle(wkx, deg=True))
        #Ex = np.abs(wkx)
        print(Ex.shape)
        #Exy_avg=Ex.mean(axis=0).mean(axis=2) # if theres not a single height, but a range
        Exy_avg=Ex.mean(axis=0) # otherwise
        print(Exy_avg.shape)
        #plt.imshow(Exy_avg)
        #plt.pcolormesh(Exy_avg)
        #plt.colorbar()
        #plt.plot(Exy_avg)
        #_data = _data.mean(["t","z"])
        #print(_data.values)
        plt.show()
        #find wavelength closest to 6000m
        idx6 = (np.abs(lmbd - 6000)).argmin()
        lmbd6 = lmbd[idx6]
        print(f'wavelength closest to 6km is: {lmbd6}')
        Exy_avg = Exy_avg[lmbd==lmbd6]
        Exy_avg = Exy_avg / 360. * lmbd6 # from phase calculate position in the real space [m]
        #print(Exy_avg)
        plt.plot(Exy_avg[0]) # phase of the 6km frequency
        #plt.xscale('log')
        #print(lmbd)
        #print(Exy_avg)
        #print(_data)
        '''        

    # if exp == 2:
    #     scale = 1e-9
    #     plt.loglog([2e3, 5e4], scale * np.array([(1./2e3)**(-5./3.), (1./5e4)**(-5./3.)]), '--', c='black', label='-5/3 scaling' )
    #     #plt.loglog(lmbd, 2e-7* K**(-5./3.) )

    for i, axc in enumerate(ax.flat):
        # subplot labels
        axc.text(
            0.05, 0.85, 
            f"({string.ascii_lowercase[i]})", 
            transform=axc.transAxes,
            fontsize=8
        )
        # Force scientific notation for any non-zero magnitude
        axc.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
        axc.invert_xaxis()

    ax[0,0].set_ylabel(params["ylabel"])
    ax[1,0].set_ylabel(params["ylabel"])
    ax[1,0].set_xlabel(r"$\lambda / 1\, \mathrm{km}$")
    ax[1,1].set_xlabel(r"$\lambda / 1\, \mathrm{km}$")
    
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=6)
    plt.savefig(figoutdir+"/spectra/"+str(outname)+"_"+str(varname)+hname+".png", dpi=300, bbox_inches='tight')
    plt.savefig(figoutdir+"/spectra/"+str(outname)+"_"+str(varname)+hname+".pdf", dpi=300, bbox_inches='tight')
    plt.show()
    plt.clf()

# save the spectra in a file using Pandas dataframe
dataframes = {}
for varname in E_lst:
    for simname in E_lst[varname]:
        if simname not in dataframes:            
            dataframes[simname] = pd.DataFrame()
            dataframes[simname]['lambda'] = E_lst[varname][simname][0]
        dataframes[simname][varname] = E_lst[varname][simname][1]

# Save the DataFrame to a CSV file
for simname in dataframes:
    output_file = dataoutdir + "/spectra_" + simname + ".csv"
    # store parameters used to compute spectra; starting with a '#' to indicate this as a comment for pandas (needs to be specified when reading!)
    with open(output_file, 'w') as file:
        file.write('# ' + str(plot_parameters) + '\n')
    # store the spectra
    dataframes[simname].to_csv(output_file, index=False, mode='a')

# plot anisitropicity
for (dx, dz, sgs), ((ts, te), p, c, d) in data_to_plot.items():
    simname = dx + ' ' + dz + ' ' + sgs
    R = (E_lst['u'][simname][1] + E['v'][simname][1]) / (2*E_lst['w'][simname][1])
    plt.loglog(E_lst['u'][simname][0], R , label=simname+"_"+varname, **p)
plt.gca().invert_xaxis()
plt.legend(bbox_to_anchor=(1.1, 1.05))
plt.ylabel("R")
plt.xlabel(r"$\lambda\, [\mathrm{m}]$")
plt.savefig(figoutdir+"/spectra/"+str(outname)+"_R_spectrum_between_"+str(height_range[0])+"m_and_"+str(height_range[1])+"m.png", dpi=300)
plt.show()
plt.clf()    

# Satellites: SEVIRI and MODIS #

## LWP Histograms ##

In [ ]:

#SEVIRI
seviri = xr.open_mfdataset(pgsdir + "/SEVIRI_CWP/ORD57328/CPP*.nc") # august 2024, Namibian region (3E-7E, 13S-17S)

# (seviri.cwp, seviri.cot, seviri.cph, '0.6-3.9 $\mu$m'),  3.9um is not used, because 1.6 is more accurate (Roebeling et al., 2006, za Kniffka et al. 2014)
# poza tym daje wartosci zupelnie inne niz 1.6 i niz MODIS (LWP 2 razy wieksze)
# zreszta Seethala 2018 tez uywa tylko 1.6
for cwp, cot, cph, label in [(seviri.cwp_16, seviri.cot_16, seviri.cph_16, 'SEVIRI 0.6-1.6 $\mu$m')]:        
    #cwp *= 1e3    #to g/m2
    cwp = cwp.where(seviri.record_status == 0) # remove points flagged as bad
    cwp = cwp.where(cot > 3) # remove thin clouds to minimize r_eff weighing (Seethala et al. 2017), doesnt make much difference 
    cwp = cwp.where(cph == 1) # remove clear-sky and ice pixels (only liquid water remains), doesnt make much difference
    
    #xr.plot.hist(cwp*1e3, bins=60, histtype='step', density=True, label=label)
    xr.plot.hist(cwp / cwp.mean(), bins=60, histtype='step', density=True, label=label)
    
#MODIS
modis_cwp = {}
modis_cwp['aqua'] = {}
modis_cwp['aqua']['default'] = xr.open_dataset(pgsdir + "/MODIS_CWP/CLDPROP_M3_MODIS_Aqua.A2024214.011.2024252000409.nc", group="/Cloud_Water_Path_Liquid/")#, engine='h5netcdf') # august 2024, L3 monthly product with histogram, entire earth
#modis_cwp['aqua']['16'] = xr.open_dataset("/home/piotr/praca/NextGEMS/LES_Smg/MODIS_CWP/CLDPROP_M3_MODIS_Aqua.A2024214.011.2024252000409.nc", group="/Cloud_Water_Path_16_Liquid/", engine='h5netcdf') 
#modis_cwp['aqua']['37'] = xr.open_dataset("/home/piotr/praca/NextGEMS/LES_Smg/MODIS_CWP/CLDPROP_M3_MODIS_Aqua.A2024214.011.2024252000409.nc", group="/Cloud_Water_Path_37_Liquid/", engine='h5netcdf') 
#modis_cwp['aqua']['PCL'] = xr.open_dataset("/home/piotr/praca/NextGEMS/LES_Smg/MODIS_CWP/CLDPROP_M3_MODIS_Aqua.A2024214.011.2024252000409.nc", group="/Cloud_Water_Path_PCL_Liquid/", engine='h5netcdf')  # PCL to partly-cloud, mniejsze LWP, olewamy
#modis_cwp['aqua']['1621'] = xr.open_dataset("/home/piotr/praca/NextGEMS/LES_Smg/MODIS_CWP/CLDPROP_M3_MODIS_Aqua.A2024214.011.2024252000409.nc", group="/Cloud_Water_Path_1621_Liquid/", engine='h5netcdf') 
modis_bins = modis_cwp['aqua']['default'].Histogram_Counts.Histogram_Bin_Boundaries

# modis cwp is cloud cells only
for key, cwp in modis_cwp['aqua'].items():
    #modis.vardim22
    #modis_cwp_roi = cwp.sel(longitude=177, latitude=77) # 177 is 3E, 77 is 13S
    modis_cwp_roi = cwp.where((cwp.longitude<=187) & (cwp.longitude >=183) & (cwp.latitude<=77) & (cwp.latitude>=73)) # 177 is 3E, 77 is 13S
    #modis_cwp_roi = cwp.where((cwp.longitude<=187) & (cwp.longitude >=183) & (cwp.latitude<=107) & (cwp.latitude>=103)) # 177 is 3E, 77 is 13S
    #modis_cwp_roi.Mean.plot(xlim=(73,77), ylim=(173,177)) # plot mean CWP in the roi
    #modis_cwp_roi = cwp
    #cwp.Mean.plot()
    #modis_cwp_roi.Mean.plot() # plot mean CWP in the roi
    #modis_cwp_roi = modis_cwp_roi 
    counts = modis_cwp_roi.Histogram_Counts.sum(dim='longitude').sum(dim='latitude') # sum over all cells in the ROI
    modis_scaled_bins = modis_bins / modis_cwp_roi.Mean.mean().values
    #pdf = counts / (np.diff(modis_bins) * counts.sum().values) # normalize the data to get a PDF, unscaled
    #plt.stairs(pdf, modis_bins, label='MODIS ' + key)
    pdf = counts / (np.diff(modis_scaled_bins) * counts.sum().values) # normalize the data to get a PDF, scaled by mean lwp
    plt.stairs(pdf, modis_scaled_bins, label='MODIS ' + key)

#UWLCM results
#bins = np.arange(5, 500, 20) # NOTE: clear-sky (<5 g/m2) excluded
#for simname in data:
#    (ts, te) = averaging_period[simname]
#    _data = data[simname]
#    _data = _data.where(_data.t>ts*3600).where(_data.t<=te*3600)
#    xr.plot.hist(_data.lwp, label=simname, bins=bins, histtype='step', density=True, **plot_params[simname])
    
#modis.Mean.plot()

#PLOT PARAMETERS
#plt.yscale('log')
#plt.xlabel('CWP [g/m$^2$]')
plt.xlabel('CWP / <CWP>')
plt.ylabel('PDF')
plt.title('satellite CWP histogram Aug. 2024 3E-7E-13S-17S')
plt.legend()
plt.xlim(-20. / 100, 500. / 100)
#plt.savefig(figoutdir+"/histograms/satellite_"+str(outname)+"_lwp_histogram.png", dpi=300)
plt.savefig(figoutdir+"/histograms/satellite_lwp_scaled_histogram.png", dpi=300)
plt.show()

## LWP spectra ##

In [ ]:
# SEVIRI
seviri = xr.open_mfdataset(pgsdir + "/SEVIRI_CWP/ORD57328/CPP*.nc") # august 2024, Namibian region (3E-7E, 13S-17S)

# MODIS
MODIS_L2_Cloud_Namibian_August2024_dir = pgsdir+"/MODIS_CWP/MODIS_CLOUD_L2_NAMIBIAN_AUGUST2024/"
MODIS_L2_Cloud_Namibian_August2024_files = [f for f in listdir(MODIS_L2_Cloud_Namibian_August2024_dir) if isfile(join(MODIS_L2_Cloud_Namibian_August2024_dir, f))]

In [ ]:
roi = (3,7,-13,-17) #3E, 7E, 13S, 17S

bins_seviri = np.logspace(np.log10(6e3), np.log10(4e5), 50) # between 6 km and 400 km
bins_modis = np.logspace(np.log10(2e3), np.log10(4e5), 50) # between 2 km and 400 km
CWP_PSD = {} # dictionary to store power spetra in seviri and modis bins

In [ ]:
import csv

# SEVIRI calculate

# (seviri.cwp, seviri.cot, seviri.cph, '0.6-3.9 $\mu$m'),  3.9um is not used, because 1.6 is more accurate (Roebeling et al., 2006, za Kniffka et al. 2014)
# poza tym daje wartosci zupelnie inne niz 1.6 i niz MODIS (LWP 2 razy wieksze)
# zreszta Seethala 2018 tez uywa tylko 1.6
for cwp, cot, cph, label in [(seviri.cwp_16, seviri.cot_16, seviri.cph_16, 'SEVIRI 0.6-1.6 $\mu$m')]:        
    cwp = cwp.where(seviri.record_status == 0) # remove points flagged as bad
    cwp = cwp.where(cot > 3) # remove thin clouds to minimize r_eff weighing (Seethala et al. 2017), doesnt make much difference 
    cwp = cwp.where(cph == 1) # remove clear-sky and ice pixels (only liquid water remains), doesnt make much difference
    cwp = cwp.dropna(dim="time", how="any") # remove framse with nan
    #cwp = cwp.fillna(0)  # replace nans (i.e. noncloudy?) with zeroes
    cwp = cwp * 1e3 # convert to g/m2

    E, K, lmbd = xrU.calc_spectrum(cwp, [1,2], 3e3, 2)
    E_avg = [E.mean(axis=0) for E in E] # time average
    E_sum = np.zeros(len(bins_seviri)-1)
    E_count = np.zeros(len(bins_seviri)-1)    
    # average over x and y
    for _E, _lmbd in zip(E_avg, lmbd):
        E_sum += binned_statistic(_lmbd, _E, statistic='sum', bins=bins_seviri).statistic
        E_count += binned_statistic(_lmbd, _E, statistic='count', bins=bins_seviri).statistic    
    CWP_PSD['SEVIRI'] = E_sum / E_count


In [ ]:
# Create a DataFrame with the bin edges and CWP_PSD_SEVIRI values
data = {
    "Bin Start": bins_seviri[:-1],
    "Bin End": bins_seviri[1:],
    "CWP_PSD_SEVIRI": CWP_PSD['SEVIRI']
}
df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
output_file = dataoutdir + "/CWP_PSD_SEVIRI.csv"
df.to_csv(output_file, index=False)

In [ ]:
# MODIS calculate
Ex_lst = []
Ey_lst = []
lmbdx_lst = []
lmbdy_lst = []

for file in MODIS_L2_Cloud_Namibian_August2024_files:
    #print(file)
    modis = xr.open_dataset(MODIS_L2_Cloud_Namibian_August2024_dir + file, engine='netcdf4')
    
    modis_roi_cf = modis.Cloud_Fraction.where((modis.Longitude<=roi[1]) & (modis.Longitude >=roi[0]) & (modis.Latitude<=roi[2]) & (modis.Latitude>=roi[3]))
    modis_roi_lat = modis.Latitude.where((modis.Longitude<=roi[1]) & (modis.Longitude >=roi[0]) & (modis.Latitude<=roi[2]) & (modis.Latitude>=roi[3]))
    modis_roi_lon = modis.Longitude.where((modis.Longitude<=roi[1]) & (modis.Longitude >=roi[0]) & (modis.Latitude<=roi[2]) & (modis.Latitude>=roi[3]))

    # check if the the entire roi is in the dataset    
    tol = 0.02 # [deg]
    if(modis_roi_lon.min() > roi[0]+tol or modis_roi_lon.max() < roi[1]-tol or modis_roi_lat.min() > roi[3]+tol or modis_roi_lat.max() < roi[2]-tol or np.isnan(modis_roi_lon.min())):
        #print("doesn't have entire roi - skipping")
        continue
    
    # find indices of cells at roi edges in order to get corresponding indices at 1km resolution
    lon = modis['Longitude'].values  # Extract longitude values
    lat = modis['Latitude'].values   # Extract latitude values
    lon_idx = [0,0]
    lat_idx = [0,0]
    for i, (target_lon, target_lat) in enumerate([(roi[1], roi[3]), (roi[0], roi[2])]):
        # Calculate the Euclidean distance between the target and all grid points
        dist = np.sqrt((lon - target_lon)**2 + (lat - target_lat)**2)
        # Find the indices of the minimum distance
        nearest_idx = np.unravel_index(np.argmin(dist), dist.shape)
        lon_idx[i] = nearest_idx[0]
        lat_idx[i] = nearest_idx[1]
#        print(f"Nearest indices: {nearest_idx}")
#        print(f"Nearest longitude: {lon[nearest_idx]}, Nearest latitude: {lat[nearest_idx]}")

    modis_roi_cf2 = modis.Cloud_Fraction[lon_idx[0]:lon_idx[1],lat_idx[0]:lat_idx[1]]
    modis_roi_cwp = modis.Cloud_Water_Path[5*lon_idx[0]:5*lon_idx[1],5*lat_idx[0]:5*lat_idx[1]] # CWP is on 1km grid, unlike 5km grid of cf

    #calculate the spectrum
    modis_roi_cwp = modis_roi_cwp.fillna(0)  # replace nans (i.e. noncloudy) with zeroes
    
    E, K, lmbd = xrU.calc_spectrum(modis_roi_cwp, [0,1], 1e3, 2)
    lmbdx_lst.append(lmbd[0])
    lmbdy_lst.append(lmbd[1])
    Ex_lst.append(E[0])
    Ey_lst.append(E[1])
    
    #Plot snapshots
    #plt.pcolormesh(modis_roi_lon, modis_roi_lat, modis_roi_cf, cmap='viridis')  # Use pcolormesh for 2D plotting
    '''
    modis_roi_cf.plot()
    plt.savefig(MODIS_L2_Cloud_Namibian_August2024_dir+'topdown/' + file + '_cf.png', dpi=300)
    plt.clf()
    
    modis_roi_cf2.plot()
    plt.savefig(MODIS_L2_Cloud_Namibian_August2024_dir+'topdown/' + file + '_cf2.png', dpi=300)
    plt.clf()
    
    modis_roi_cwp.plot()
    plt.savefig(MODIS_L2_Cloud_Namibian_August2024_dir+'topdown/' + file + '_cwp.png', dpi=300)
    plt.clf()
    '''
    '''
    modis_roi_lat.plot()
    plt.savefig(MODIS_L2_Cloud_Namibian_August2024_dir+'topdown/' + file + '_lat.png', dpi=300)
    plt.clf()
    modis_roi_lon.plot()
    plt.savefig(MODIS_L2_Cloud_Namibian_August2024_dir+'topdown/' + file + '_lon.png', dpi=300)
    plt.clf()
    '''

In [ ]:
E_sum = np.zeros(len(bins_modis)-1)
E_count = np.zeros(len(bins_modis)-1)    

# average over x and y and over time
for _E_t, _lmbd_t in zip(Ex_lst + Ey_lst, lmbdx_lst + lmbdy_lst):
    for(_E, _lmbd) in zip(_E_t, _lmbd_t):
        E_sum += binned_statistic(_lmbd, _E, statistic='sum', bins=bins_modis).statistic
        E_count += binned_statistic(_lmbd, _E, statistic='count', bins=bins_modis).statistic    

CWP_PSD['MODIS'] = E_sum / E_count

In [ ]:
# Save CWP_PSD and bins_modis to a CSV file
# Create a DataFrame with the bin edges and CWP_PSD_MODIS values
data = {
    "Bin Start": bins_modis[:-1],
    "Bin End": bins_modis[1:],
    "CWP_PSD_MODIS": CWP_PSD['MODIS']
}
df = pd.DataFrame(data)

# Save the DataFrame to a CSV file
output_file = dataoutdir + "/CWP_PSD_MODIS.csv"
df.to_csv(output_file, index=False)

# Plot LWP/CWP spectra 

In [ ]:
# Load satellite spectra from the CSV file
# scale them so that they are =1 for 10km and shifted upwards for clarity
shift = 1
for i, sat in enumerate(["SEVIRI", "MODIS"]):
    input_file = dataoutdir + "/CWP_PSD_"+sat+".csv"
    df = pd.read_csv(input_file)
    bins = df["Bin Start"].tolist()
    bins.append(df["Bin End"].iloc[-1].tolist())
    scaleidx = np.absolute(np.array(bins)-1e4).argmin()
    psd = np.array(df["CWP_PSD_"+sat].tolist())
    psd = psd / psd[scaleidx] * shift**(i)
    #plt.stairs(df["CWP_PSD_"+sat].tolist(), bins, label=sat + ' CWP')
    plt.stairs(psd, bins, label=sat + ' CWP shifted')

# Load simulation spectra from the CSV file
for i, ((dx, dz, sgs), ((ts, te), p, c, d)) in enumerate(data_to_plot.items()):
    simname = dx + ' ' + dz + ' ' + sgs
    input_file = dataoutdir + "/spectra_" + simname + ".csv"
    df = pd.read_csv(input_file, comment='#')
    lmbd = np.array(df['lambda'].tolist())
    scaleidx = np.absolute(lmbd-1e4).argmin()
    psd = np.array(df['lwp'].tolist())
    psd = psd / psd[scaleidx] * shift**(i+2)
    #plt.loglog(df['lambda'], df['lwp'])
    plt.loglog(lmbd, psd)

plt.loglog([2e3, 5e4], [1e-6* (1./2e3)**(-5./3.), 1e-6*(1./5e4)**(-5./3.)], '--', c='black', label='-5/3 scaling' )
plt.yscale('log')
plt.xscale('log')
plt.gca().invert_xaxis()
plt.legend()
plt.savefig(figoutdir+"/spectra/lwp_spectra_"+outname+".png", dpi=300)